# Fairness adversarial perturbation on CelebA

A smile classifier is trained on a deliberately skewed slice of CelebA, in
which 75% of the smiling faces are female. The classifier picks up gender as a
shortcut, and its error rate splits along gender lines.

FAAP fixes that without touching the classifier. A generator learns a small,
image-dependent perturbation such that a discriminator reading the classifier's
own 512-d penultimate features can no longer tell male from female, while the
smile prediction survives. The deployed model is unchanged; only its input is.

Three things here differ from the usual write-up of this experiment, and all
three change what the numbers mean:

* the generator's output is bounded by `eps * tanh(.)` rather than by clipping
  a `tanh`, so it stays differentiable everywhere;
* the generator minimises cross-entropy against a **uniform** target, which is
  attained when the features carry no gender information, rather than the
  **negated** cross-entropy, which is minimised by making the discriminator
  confidently wrong;
* attribute leakage is measured on held-out data, not on the batch the
  discriminator was just updated on.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset

import trustlab as tl

tl.set_seed(tl.SEED)
DEVICE = tl.DEVICE
print("device:", DEVICE)

In [ ]:
BATCH_SIZE = 64
IMAGE_SIZE = 224
CLASSIFIER_EPOCHS = 10
CLASSIFIER_LR = 5e-4

FAAP_EPOCHS = 10
GAN_LR = 2e-4
EPSILON = 0.10          # in [-1, 1] units; see below
ALPHA = 0.1             # weight on the entropy term
BETA = 1.0              # weight on keeping the smile task intact

print(f"perturbation budget: eps = {EPSILON} in [-1, 1] units "
      f"= {tl.eps_to_pixels(EPSILON):.2f}/255 in pixel levels")

## Data

CelebA is normalised with mean = std = 0.5, so images live in `[-1, 1]`. The
budget above is therefore **12.75/255** in pixel terms --- large by the
standards of adversarial robustness work, where 8/255 is the usual ceiling, and
worth stating in those units rather than leaving it as a bare 0.1.

Three disjoint splits are drawn, each with its own subgroup quota:

| split | size | composition |
|---|---|---|
| biased classifier train | 4,000 | 45% female-smiling, 15% male-smiling --- the skew |
| FAAP (generator) train | 4,000 | drawn from what is left, no quota imposed |
| test | 2,000 | drawn from what is left, no quota imposed |

Only the first is skewed. The generator and the evaluation see whatever the
dataset's own balance gives, so the fairness numbers are not measured on the
same distortion that created the bias.

In [ ]:
train_pool = tl.celeba(split="train", image_size=IMAGE_SIZE)
buckets = tl.subgroup_indices(train_pool)
print({k: len(v) for k, v in buckets.items()})

In [ ]:
rng = np.random.default_rng(tl.SEED)

BIASED_QUOTAS = {"female_smile": 1800, "male_smile": 600,
                 "female_nosmile": 800, "male_nosmile": 800}
biased_idx = tl.biased_split(buckets, BIASED_QUOTAS, rng=rng)

# the generator and test splits come from what the biased split did not take
used = set(int(i) for i in biased_idx)
remaining = {g: np.array([i for i in v if int(i) not in used])
             for g, v in buckets.items()}
rest = np.concatenate(list(remaining.values()))
rng.shuffle(rest)
faap_idx, test_idx = rest[:4000], rest[4000:6000]

for name, idx in [("Biased Classifier Train Set", biased_idx),
                  ("FAAP (GAN) Set", faap_idx),
                  ("Test Set", test_idx)]:
    table, _ = tl.subgroup_table(buckets, idx, name=name)
    print(table, "\n")

In [ ]:
biased_loader = DataLoader(Subset(train_pool, biased_idx), batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=0)
faap_loader = DataLoader(Subset(train_pool, faap_idx), batch_size=BATCH_SIZE,
                         shuffle=True, num_workers=0)
test_loader = DataLoader(Subset(train_pool, test_idx), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=0)

In [ ]:
batch = next(iter(biased_loader))
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, img, smile, male in zip(axes.ravel(), batch["image"],
                                batch["Smiling"], batch["Male"]):
    ax.imshow((img.permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1).numpy())
    ax.set_title(f"{'Male' if male else 'Female'}, "
                 f"{'Smiling' if smile else 'Not smiling'}", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## The biased classifier

ResNet-18 from scratch, 10 epochs, on the skewed 4,000. Nothing about this
model is later modified: FAAP treats it as a black box that it is allowed to
read gradients through but not to change.

In [ ]:
model = tl.SmileClassifier(num_classes=2).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=CLASSIFIER_LR)

history = {"loss": [], "acc": []}
for epoch in range(CLASSIFIER_EPOCHS):
    model.train()
    running, correct, total = 0.0, 0, 0
    for batch in biased_loader:
        images = batch["image"].to(DEVICE)
        labels = batch["Smiling"].to(DEVICE).long()
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    history["loss"].append(running / total)
    history["acc"].append(100.0 * correct / total)
    print(f"epoch {epoch+1:2d}/{CLASSIFIER_EPOCHS}  "
          f"loss {history['loss'][-1]:.4f}  acc {history['acc'][-1]:.2f}%")

torch.save(model.state_dict(), "biased_classifier.pth")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(history["loss"], marker="o"); ax1.set_title("training loss")
ax2.plot(history["acc"], marker="o"); ax2.set_title("training accuracy (%)")
for ax in (ax1, ax2):
    ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
baseline = tl.fairness_metrics(model, test_loader)
print(f"test accuracy      {baseline['accuracy']:.4f}")
print(f"DP difference      {baseline['dp_diff']:.4f}")
print(f"EO difference      {baseline['eo_diff']:.4f}")
print(f"positive rate      female {baseline['positive_rate'][0]:.4f}  "
      f"male {baseline['positive_rate'][1]:.4f}")
print(f"TPR                female {baseline['tpr'][0]:.4f}  "
      f"male {baseline['tpr'][1]:.4f}")

## FAAP

The classifier is frozen. Two networks are trained against each other on the
generator split:

* the **discriminator** tries to predict gender from the frozen classifier's
  512-d features of the *perturbed* image;
* the **generator** produces the perturbation, and wants the discriminator at
  chance while the smile logits stay correct.

The generator's objective is

```
L_G = confusion(D(f(x + delta)))  -  ALPHA * H(D(...))  +  BETA * CE(head(f(x + delta)), y_smile)
```

where `confusion` is cross-entropy against a uniform target. Both it and the
entropy term are maximised at the same place --- a discriminator that outputs
0.5, 0.5 --- so they reinforce rather than fight; the entropy term simply
sharpens the gradient near the optimum.

In [ ]:
feature_extractor = model.feature_extractor
label_predictor = model.classifier_head
for p in model.parameters():
    p.requires_grad_(False)
feature_extractor.eval()
label_predictor.eval()

generator = tl.Generator(eps=EPSILON).to(DEVICE)
discriminator = tl.LatentDiscriminator(model.num_features).to(DEVICE)

optimizer_G = torch.optim.Adam(generator.parameters(), lr=GAN_LR, betas=(0.5, 0.999))
optimizer_D = torch.optim.Adam(discriminator.parameters(), lr=GAN_LR, betas=(0.5, 0.999))
criterion_task = nn.CrossEntropyLoss()
criterion_fair = nn.CrossEntropyLoss()

In [ ]:
log = {"loss_D": [], "loss_G": [], "loss_task": [], "leakage": []}

for epoch in range(FAAP_EPOCHS):
    generator.train()
    discriminator.train()
    sums = {"D": 0.0, "G": 0.0, "task": 0.0}
    n_batches = 0

    for batch in faap_loader:
        images = batch["image"].to(DEVICE)
        labels_gender = batch["Male"].to(DEVICE).long()
        labels_smile = batch["Smiling"].to(DEVICE).long()

        # --- discriminator: read gender out of the perturbed features
        optimizer_D.zero_grad()
        with torch.no_grad():
            perturbed = tl.perturb(generator, images)
        loss_D = criterion_fair(discriminator(feature_extractor(perturbed)),
                                labels_gender)
        loss_D.backward()
        optimizer_D.step()

        # --- generator: drive the discriminator to chance, keep the smile
        optimizer_G.zero_grad()
        perturbed = tl.perturb(generator, images)
        features = feature_extractor(perturbed)
        logits_gender = discriminator(features)
        logits_smile = label_predictor(features)

        probs = torch.softmax(logits_gender, dim=1)
        entropy = -(probs * torch.log(probs + 1e-8)).sum(dim=1).mean()

        loss_G_fair = tl.confusion_loss(logits_gender) - ALPHA * entropy
        loss_G_task = criterion_task(logits_smile, labels_smile)
        loss_G = loss_G_fair + BETA * loss_G_task
        loss_G.backward()
        optimizer_G.step()

        sums["D"] += loss_D.item()
        sums["G"] += loss_G.item()
        sums["task"] += loss_G_task.item()
        n_batches += 1

    leakage = tl.attribute_leakage(discriminator, feature_extractor,
                                   test_loader, generator=generator)
    log["loss_D"].append(sums["D"] / n_batches)
    log["loss_G"].append(sums["G"] / n_batches)
    log["loss_task"].append(sums["task"] / n_batches)
    log["leakage"].append(leakage)

    print(f"epoch {epoch+1:2d}/{FAAP_EPOCHS}  "
          f"D {log['loss_D'][-1]:.4f}  G {log['loss_G'][-1]:.4f}  "
          f"task {log['loss_task'][-1]:.4f}  "
          f"held-out gender leakage {100*leakage:.2f}%  (chance 50%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(log["loss_D"], marker="o"); axes[0].set_title("discriminator loss")
axes[1].plot(log["loss_G"], marker="o", label="total")
axes[1].plot(log["loss_task"], marker="s", label="smile task")
axes[1].set_title("generator loss"); axes[1].legend()
axes[2].plot([100 * v for v in log["leakage"]], marker="o")
axes[2].axhline(50, ls="--", c="r", label="chance")
axes[2].set_title("held-out gender leakage (%)"); axes[2].legend()
for ax in axes:
    ax.set_xlabel("epoch"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
generator.eval()
batch = next(iter(test_loader))
images = batch["image"][:5].to(DEVICE)
with torch.no_grad():
    delta = generator(images)
    perturbed = torch.clamp(images + delta, -1, 1)

def show(t):
    return (t.detach().cpu().permute(1, 2, 0) * 0.5 + 0.5).clamp(0, 1).numpy()

fig, axes = plt.subplots(3, 5, figsize=(14, 9))
for i in range(5):
    axes[0, i].imshow(show(images[i]))
    axes[1, i].imshow(show(perturbed[i]))
    # the perturbation itself, rescaled to be visible at all
    d = delta[i].detach().cpu()
    axes[2, i].imshow(((d - d.min()) / (d.max() - d.min() + 1e-8)).permute(1, 2, 0).numpy())
    for r in range(3):
        axes[r, i].axis("off")
axes[0, 0].set_title("original", loc="left")
axes[1, 0].set_title("perturbed", loc="left")
axes[2, 0].set_title(f"perturbation (x{1/EPSILON:.0f}, rescaled)", loc="left")
plt.tight_layout(); plt.show()

print(f"max |delta| = {delta.abs().max().item():.4f}  (budget {EPSILON})")

## Evaluation

Both rows below use the **same untouched classifier**; the only difference is
whether its input was perturbed. That is the whole claim FAAP makes, so
evaluating any other way --- retraining, fine-tuning, swapping the head ---
would be measuring something else.

In [ ]:
after = tl.fairness_metrics(model, test_loader, generator=generator)

rows = [("Accuracy", baseline["accuracy"], after["accuracy"]),
        ("DP diff", baseline["dp_diff"], after["dp_diff"]),
        ("EO diff", baseline["eo_diff"], after["eo_diff"])]
print(f"{'metric':<10} {'before':>8} {'after':>8} {'change':>9}")
for name, b, a in rows:
    print(f"{name:<10} {b:8.3f} {a:8.3f} {100*(a-b)/b:8.1f}%")

print()
for tag, m in [("before", baseline), ("after", after)]:
    print(f"{tag:<7} positive rate  female {m['positive_rate'][0]:.3f}  "
          f"male {m['positive_rate'][1]:.3f}   "
          f"TPR  female {m['tpr'][0]:.3f}  male {m['tpr'][1]:.3f}")

In [ ]:
from sklearn.manifold import TSNE

@torch.no_grad()
def latent(loader, gen=None):
    feats, smiles, genders = [], [], []
    for batch in loader:
        images = batch["image"].to(DEVICE)
        if gen is not None:
            images = tl.perturb(gen, images)
        feats.append(feature_extractor(images).cpu().numpy())
        smiles.append(batch["Smiling"].numpy())
        genders.append(batch["Male"].numpy())
    return (np.concatenate(feats), np.concatenate(smiles), np.concatenate(genders))

feat_before, smile_lab, gender_lab = latent(test_loader)
feat_after, _, _ = latent(test_loader, gen=generator)

class_names = ["Female, not smiling", "Female, smiling",
               "Male, not smiling", "Male, smiling"]
combined = gender_lab * 2 + smile_lab

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, feats, title in [(axes[0], feat_before, "before FAAP"),
                         (axes[1], feat_after, "after FAAP")]:
    # max_iter, not n_iter: renamed in scikit-learn 1.5, removed in 1.7.
    # 1000 is the default and the embedding needs it; 300 stops short.
    emb = TSNE(n_components=2, random_state=tl.SEED, perplexity=30,
               max_iter=1000, init="pca").fit_transform(feats)
    for c, name in enumerate(class_names):
        mask = combined == c
        ax.scatter(emb[mask, 0], emb[mask, 1], s=6, alpha=0.6, label=name)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
axes[1].legend(markerscale=3, fontsize=9)
plt.tight_layout(); plt.show()

## Reading the result

The accuracy going **up** under an adversarial perturbation is the part worth
being careful about. It is not impossible --- the perturbation is trained with
`BETA = 1.0` explicitly pushing on the smile loss, so it can act as a learned
input-space correction --- but a defence that improves the metric it was
supposed to trade against is exactly the shape of a measurement error, and it
should be read together with the leakage curve rather than on its own.

The leakage number is the honest one to watch. If it does not approach 50% the
gender information is still in the features and the fairness gain came from
somewhere else: the perturbation shifting the decision threshold, or the two
groups' error rates moving together by luck. The original reported a *training*
fooling accuracy instead, measured on the same batch the discriminator had just
been updated on, and it fell from 42% to 36% over the ten epochs --- the
discriminator winning --- while the write-up described the metrics as
"constant and at an acceptable level".